In [1]:
import pandas as pd
import numpy as np

# Configuração do ambiente e constantes
USER = "lpvianna81"
REPO = "projetofinal_tecnicas_python"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{USER}/{REPO}/{BRANCH}/"

def executar_pipeline_f1():

    try:
        # Carregamos os dados e realizamos os joins fundamentais
        results = pd.read_csv(f"{BASE_URL}results.csv")
        status = pd.read_csv(f"{BASE_URL}status.csv")

        # Renomeação preventiva para evitar conflitos de nomes comuns ('name', 'nationality')
        drivers = pd.read_csv(f"{BASE_URL}drivers.csv").rename(columns={'nationality': 'driver_nationality'})
        drivers['driver_full_name'] = drivers['forename'] + ' ' + drivers['surname']

        races = pd.read_csv(f"{BASE_URL}races.csv").rename(columns={'name': 'race_name'})

        constructors = pd.read_csv(f"{BASE_URL}constructors.csv").rename(columns={
            'name': 'team_name',
            'nationality': 'team_nationality'
        })

        # Merge central (Star Schema style)
        df = results.merge(drivers, on='driverId') \
                    .merge(races, on='raceId') \
                    .merge(constructors, on='constructorId') \
                    .merge(status, on='statusId')

        # Tratamento de nulos padrão da F1 (\N) e conversão de tipos
        df.replace(r'\N', np.nan, inplace=True)

        df['points'] = pd.to_numeric(df['points'], errors='coerce').fillna(0)
        df['grid'] = pd.to_numeric(df['grid'], errors='coerce')
        df['positionOrder'] = pd.to_numeric(df['positionOrder'], errors='coerce')
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['dob'] = pd.to_datetime(df['dob'], errors='coerce')

        print("Pipeline: Dados integrados e higienizados.\n")

        print("="*60)
        print("RELATÓRIO TÉCNICO DE PERFORMANCE F1")
        print("="*60)

        # Q1
        print("\n1. Top 10 Pilotos com mais vitórias:")
        print(df[df['positionOrder'] == 1]['driver_full_name'].value_counts().head(10))

        # Q2
        print("\n2. Top 5 Equipes com mais pontos:")
        print(df.groupby('team_name')['points'].sum().sort_values(ascending=False).head(5))

        # Q3
        print("\n3. Pilotos com mais Pole Positions:")
        print(df[df['grid'] == 1]['driver_full_name'].value_counts().head(10))

        # Q4
        print("\n4. Circuitos com mais GPs realizados:")
        print(df.groupby('race_name')['raceId'].nunique().sort_values(ascending=False).head(10))

        # Q5
        print("\n5. Recordista de Voltas Mais Rápidas (Rank 1):")
        print(df[df['rank'] == '1']['driver_full_name'].value_counts().head(1))

        # Q6
        print("\n6. Número de vitórias de pilotos por nacionalidade:")
        print(df[df['positionOrder'] == 1]['driver_nationality'].value_counts())

        # Q7
        ultimo_ano = df['year'].max()
        ferrari = df[(df['team_name'] == 'Ferrari') & (df['year'] > (ultimo_ano - 20))]
        print("\n7. Evolução de pontos da Ferrari (Últimas 20 temporadas):")
        print(ferrari.groupby('year')['points'].sum())

        # Q8
        br_vits = df[(df['driver_nationality'] == 'Brazilian') & (df['positionOrder'] == 1)]
        print("\n8. Pilotos Brasileiros com mais vitórias:")
        print(br_vits['driver_full_name'].value_counts().head(3))

        # Q9
        print("\n9. Pilotos com mais GPs disputados na carreira:")
        print(df['driver_full_name'].value_counts().head(10))

        # Q10: Média de idade dos campeões por década
        # Primeiro, precisamos agregar pontos por ano/piloto para definir o campeão
        season_agg = df.groupby(['year', 'driverId', 'dob'])['points'].sum().reset_index()
        # Encontramos o ID do campeão de cada ano
        idx_champs = season_agg.groupby(['year'])['points'].idxmax()
        champs_df = season_agg.loc[idx_champs].copy()

        # Cálculos de idade e década
        champs_df['age'] = champs_df['year'] - champs_df['dob'].dt.year
        champs_df['decade'] = (champs_df['year'] // 10) * 10

        print("\n10. Média de idade dos campeões mundiais por década:")
        print(champs_df.groupby('decade')['age'].mean())

    except Exception as e:
        print(f"Erro no Pipeline: {e}")

if __name__ == "__main__":
    executar_pipeline_f1()

Pipeline: Dados integrados e higienizados.

RELATÓRIO TÉCNICO DE PERFORMANCE F1

1. Top 10 Pilotos com mais vitórias:
driver_full_name
Lewis Hamilton        105
Michael Schumacher     91
Max Verstappen         63
Sebastian Vettel       53
Alain Prost            51
Ayrton Senna           41
Fernando Alonso        32
Nigel Mansell          31
Jackie Stewart         27
Niki Lauda             25
Name: count, dtype: int64

2. Top 5 Equipes com mais pontos:
team_name
Ferrari     11091.27
Mercedes     7730.64
Red Bull     7673.00
McLaren      7022.50
Williams     3641.00
Name: points, dtype: float64

3. Pilotos com mais Pole Positions:
driver_full_name
Lewis Hamilton        104
Michael Schumacher     68
Ayrton Senna           65
Sebastian Vettel       57
Max Verstappen         40
Jim Clark              34
Alain Prost            33
Nigel Mansell          32
Nico Rosberg           30
Juan Fangio            29
Name: count, dtype: int64

4. Circuitos com mais GPs realizados:
race_name
British Gra